# Crop production (Vietnam Mekong)

Notebook to read RIBASIM HIS crop-production files and store them as **Zarr**.

Source files (see `read_ribasim_his.ipynb` in Food-Security):

- `RIB_CULT_prod.his` — crop production (`path` in TOML)
- `RIB_ADVIR_dmnd.his` — crop hectares per season (`ha_path` in TOML)

Outputs live alongside the salinity products under:
`N:\...\stac_folder\crop_production\` (see `11_salinity.ipynb`).

**Zarr-safe variable names:** HIS names can contain `/` or `#` (e.g. `P Cr02/WinterSpring (ha)`).
`/` is a path separator in Zarr and silently breaks those arrays. This notebook replaces `/`
with `_` before writing, stores the rename map in dataset attrs (`HIS_VAR_NAME_MAP`) and a
sidecar JSON (`*_var_name_map.json`), so consumers can restore original HIS names when reading.

**Prerequisites**

1. `mamba activate coclico`
2. `pip install -e .` in Food-Security (or use `sys.path` workaround below)
3. `metadata_crop_production.json` present in `data_dir`

In [13]:
# Optional; code formatter
# %load_ext nb_black

### Configure paths and imports

In [ ]:
import datetime
import json
import os
import sys
from pathlib import Path

import xarray as xr

sys.path.insert(0, r"C:\Ocean\Work\Projects\2026\IDP\Tools\Food-Security")
from food_security import data_reader

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [15]:
repo_root = Path(r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories")

data_dir = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production"
)
metadata_path = data_dir / "metadata_crop_production.json"

data_dir.mkdir(parents=True, exist_ok=True)

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

with open(metadata_path) as f:
    metadata = json.load(f)

prod_out_file = "RIB_CULT_prod"
ha_out_file = "RIB_ADVIR_dmnd"

prod_zarr = data_dir / f"{prod_out_file}.zarr"
ha_zarr = data_dir / f"{ha_out_file}.zarr"

PROD_HIS = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Ribasim8_MekDelta26"
    r"/Modules/ribasim/Mekong.1/work/RIB_CULT_prod.his"
)
HA_HIS = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Ribasim8_MekDelta26"
    r"/Modules/ribasim/Mekong.1/work/RIB_ADVIR_dmnd.his"
)

print("Output dir:", data_dir)
print("Metadata:", metadata_path)
for label, path in [("Production HIS", PROD_HIS), ("Hectares HIS", HA_HIS)]:
    print(f"{label}: {path}")
    print(f"  exists: {path.is_file()}")

Output dir: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production
Metadata: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production\metadata_crop_production.json
Production HIS: P:\11211454-002-idt\IDP\Vietnam\Ribasim8_MekDelta26\Modules\ribasim\Mekong.1\work\RIB_CULT_prod.his
  exists: True
Hectares HIS: P:\11211454-002-idt\IDP\Vietnam\Ribasim8_MekDelta26\Modules\ribasim\Mekong.1\work\RIB_ADVIR_dmnd.his
  exists: True


### Read RIBASIM HIS files

In [16]:
def read_his(path: Path, *, use_hia: bool = False) -> xr.Dataset:
    """Read a RIBASIM HIS file and return the xarray Dataset."""
    reader = data_reader.HisFile(path, crop=None)
    reader.read(hia=use_hia)
    return reader.ds


def sanitize_zarr_var_name(name: str) -> str:
    """Make a HIS variable name safe for Zarr group keys.

    Replaces characters that break Zarr/fsspec paths:
    - '/' (path separator)
    - '#' (URL/fragment delimiter; can zero-out arrays on remote stores)
    """
    return name.replace("/", "_").replace("#", "num")


def make_zarr_safe(ds: xr.Dataset) -> tuple[xr.Dataset, dict[str, str]]:
    """Rename data vars so Zarr can store them; return (ds, original->safe map)."""
    mapping: dict[str, str] = {}
    used: set[str] = set()
    for name in ds.data_vars:
        safe = sanitize_zarr_var_name(name)
        base = safe
        suffix = 2
        while safe in used:
            safe = f"{base}__{suffix}"
            suffix += 1
        used.add(safe)
        if safe != name:
            mapping[name] = safe
    out = ds.rename(mapping) if mapping else ds.copy(deep=True)
    return out, mapping


def restore_his_var_names(
    ds: xr.Dataset, safe_to_original: dict[str, str] | None = None
) -> xr.Dataset:
    """Rename Zarr-safe vars back to original HIS names."""
    if safe_to_original is None:
        raw = ds.attrs.get("HIS_VAR_NAME_MAP")
        if not raw:
            return ds
        safe_to_original = json.loads(raw)
    rename = {k: v for k, v in safe_to_original.items() if k in ds.data_vars}
    return ds.rename(rename) if rename else ds


def save_var_name_map(path: Path, original_to_safe: dict[str, str]) -> None:
    payload = {
        "original_to_safe": original_to_safe,
        "safe_to_original": {v: k for k, v in original_to_safe.items()},
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("Wrote var name map:", path)


def dataset_for_zarr(
    ds: xr.Dataset, metadata: dict
) -> tuple[xr.Dataset, dict[str, str]]:
    """Prepare dataset for Zarr: sanitize '/' in names + CF/metadata attrs.

    Returns (zarr-ready dataset, original->safe rename map).
    """
    out, mapping = make_zarr_safe(ds)
    for key, value in list(out.attrs.items()):
        if isinstance(value, datetime.datetime):
            out.attrs[key] = value.isoformat()
    for attr_name, attr_val in metadata.items():
        if attr_name == "PROVIDERS":
            attr_val = json.dumps(attr_val)
        out.attrs[attr_name] = attr_val
    out.attrs["Conventions"] = "CF-1.8"
    # safe -> original, so readers can restore HIS names
    out.attrs["HIS_VAR_NAME_MAP"] = json.dumps({v: k for k, v in mapping.items()})
    out["time"].attrs.update(
        {"long_name": "Time", "standard_name": "time", "axis": "T"}
    )
    return out, mapping


def dataset_summary(ds: xr.Dataset, title: str) -> None:
    print(title)
    print("=" * len(title))
    print(f"Time steps : {len(ds.time)} ({ds.time.values[0]} -> {ds.time.values[-1]})")
    print(f"Stations   : {len(ds.station)}")
    print(f"Variables  : {len(ds.data_vars)}")


prod_ds = read_his(PROD_HIS, use_hia=False)
ha_ds = read_his(HA_HIS, use_hia=True)

dataset_summary(prod_ds, "RIB_CULT_prod.his")
print()
dataset_summary(ha_ds, "RIB_ADVIR_dmnd.his")


RIB_CULT_prod.his
Time steps : 3 (2014-01-01T00:00:00.000000000 -> 2016-01-01T00:00:00.000000000)
Stations   : 48
Variables  : 23

RIB_ADVIR_dmnd.his
Time steps : 72 (2014-01-01T00:00:00.000000000 -> 2016-12-16T00:00:00.000000000)
Stations   : 12
Variables  : 37


## 1. `RIB_CULT_prod.his` → Zarr

In [17]:
prod_out, prod_name_map = dataset_for_zarr(prod_ds, metadata)
prod_out.to_zarr(prod_zarr, mode="w")
save_var_name_map(data_dir / f"{prod_out_file}_var_name_map.json", prod_name_map)
print(f"Renamed {len(prod_name_map)} production variables for Zarr safety")

Wrote var name map: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production\RIB_CULT_prod_var_name_map.json
Renamed 0 production variables for Zarr safety


In [18]:
prod_check = restore_his_var_names(xr.open_zarr(prod_zarr))
print(
    "Variables preserved (after restore):",
    set(prod_check.data_vars) == set(prod_ds.data_vars),
)
print("Count HIS / restored:", len(prod_ds.data_vars), len(prod_check.data_vars))
prod_check

Variables preserved (after restore): True
Count HIS / restored: 23 23


<xarray.Dataset> Size: 17kB
Dimensions:                (time: 3, station: 48)
Coordinates:
  * time                   (time) datetime64[ns] 24B 2014-01-01 ... 2016-01-01
  * station                (station) <U20 4kB 'Nd______42 / Cr__3 /' ... 'Nd_...
Data variables: (12/23)
    + Actual rainfall (    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    + Allocation (Mcm)     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    + Decrease RZ SM +     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    - Act.evapotranspir    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    - Act.percolation (    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    - Increase RZ SM +     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    ...                     ...
    Potent.farm gate pr    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.farm gate pr_2  (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.field level     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.field level_2   (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.production c    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Survival fraction (    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
Attributes: (12/33)
    AUTHOR:              Author
    CITATION:            Citation
    COLLECTION_ID:       crop_production
    COMMENT:             
    CRS:                 EPSG:32648
    Conventions:         CF-1.8
    ...                  ...
    TITLE_ABBREVIATION:  Abbreviation
    UNITS:                Units
    WMS_DATASET:         crop_production
    header:              All cultivations                        Yearly agric...
    scu:                 86400
    t0:                  2014-01-01T00:00:00

## 2. `RIB_ADVIR_dmnd.his` → Zarr

In [19]:
ha_out, ha_name_map = dataset_for_zarr(ha_ds, metadata)
ha_out.to_zarr(ha_zarr, mode="w")
save_var_name_map(data_dir / f"{ha_out_file}_var_name_map.json", ha_name_map)
print(f"Renamed {len(ha_name_map)} hectare variables for Zarr safety")
if ha_name_map:
    print("Examples:", list(ha_name_map.items())[:5])

Wrote var name map: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production\RIB_ADVIR_dmnd_var_name_map.json
Renamed 20 hectare variables for Zarr safety
Examples: [('Field crop water requirements (mm/day)', 'Field crop water requirements (mm_day)'), ('Field crop water requirements (l/s/ha)', 'Field crop water requirements (l_s_ha)'), ('Demand if feedback (mm/day)', 'Demand if feedback (mm_day)'), ('Demand if feedback (l/s/ha)', 'Demand if feedback (l_s_ha)'), ('Supply (mm/day)', 'Supply (mm_day)')]


In [20]:
ha_check = restore_his_var_names(xr.open_zarr(ha_zarr))
print(
    "Variables preserved (after restore):",
    set(ha_check.data_vars) == set(ha_ds.data_vars),
)
print("Count HIS / restored:", len(ha_ds.data_vars), len(ha_check.data_vars))
ha_check

Variables preserved (after restore): True
Count HIS / restored: 37 37


<xarray.Dataset> Size: 130kB
Dimensions:                                              (time: 72, station: 12)
Coordinates:
  * time                                                 (time) datetime64[ns] 576B ...
  * station                                              (station) <U23 1kB '...
Data variables: (12/37)
    + Actual rainfall (Mcm)                              (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    + Decrease RZ SM + S field (Mcm)                     (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    + Gross water supply (Mcm)                           (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    - Actual evapotranspiration (Mcm)                    (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    - Actual percolation (Mcm)                           (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    - Drainage from fields (Mcm)                         (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    ...                                                   ...
    Shortage per time step (# of times)                  (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Supply (l/s/ha)                                      (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Supply (mm/day)                                      (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Supply from network (m3/s)                           (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Supply-demand ratio (%)                              (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Water balance term (must be 0.0 during grow.season)  (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
Attributes: (12/33)
    AUTHOR:              Author
    CITATION:            Citation
    COLLECTION_ID:       crop_production
    COMMENT:             
    CRS:                 EPSG:32648
    Conventions:         CF-1.8
    ...                  ...
    TITLE_ABBREVIATION:  Abbreviation
    UNITS:                Units
    WMS_DATASET:         crop_production
    header:              Advanced irrigation nodes               Demand and a...
    scu:                 86400
    t0:                  2014-01-01T00:00:00